In [17]:
import numpy as np
import os
import re

def load_viewpoint_poses(folder_path):
    """
    Loads all .npy pose files from a directory into a list, 
    sorted numerically by the index in the filename.
    """
    # Regex to capture the index number from 'viewpoint_pose_X.npy'
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1

    # Filter for .npy files and sort them (0, 1, 2... instead of 0, 1, 10...)
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    npy_files.sort(key=extract_number)

    # Load data into list
    viewpoint_poses = []
    for file_name in npy_files:
        full_path = os.path.join(folder_path, file_name)
        try:
            pose = np.load(full_path)
            viewpoint_poses.append(pose)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")

    return viewpoint_poses

# Usage
path = r"viewpoints_candidate"
viewpoint_poses = load_viewpoint_poses(path)

print(f"Loaded {len(viewpoint_poses)} matrices.")
if viewpoint_poses:
    print("Example of first matrix:\n", viewpoint_poses[0])

print('sd')

Loaded 27 matrices.
Example of first matrix:
 [[-8.84353258e-01  9.53997077e-02 -4.56966312e-01  1.77500000e+02]
 [ 1.04083409e-16  9.78895479e-01  2.04361547e-01 -7.44427191e+01]
 [ 4.66818288e-01  1.80727800e-01 -8.65689406e-01  4.16885438e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
sd


In [123]:
import open3d as o3d
import numpy as np
from scipy.spatial.transform import Rotation as R


def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] in degrees to a 4x4 transformation matrix.
    If zyz=True, assumes the input is in ZYZ Euler Angles, otherwise XYZ Euler Angles.
    
    Args:
    - pose (list or array): [x, y, z, roll, pitch, yaw] in degrees.
    - zyz (bool): If True, interprets the last three values as ZYZ Euler angles. 
                  If False, uses XYZ Euler angles (default).
    
    Returns:
    - T (ndarray): The 4x4 homogeneous transformation matrix.
    """
    x, y, z = pose[:3]   # Translation vector
    roll, pitch, yaw = pose[3:]  # Rotation angles in degrees

    # Convert XYZ Euler Angles to a 3x3 rotation matrix
    rot_matrix = R.from_euler('XYZ', [roll, pitch, yaw], degrees=True).as_matrix()

    # Build the 4x4 Homogeneous Transformation Matrix
    T = np.eye(4)  # Start with the identity matrix (4x4)
    T[:3, :3] = rot_matrix  # Set the upper-left 3x3 part to the rotation matrix
    T[:3, 3] = [x, y, z]    # Set the upper-right 3x1 part to the translation vector
    return T

def visualize_transforms(matrices, axis_size=50.0):
    """
    Visualizes a single 4x4 matrix or a list/array of 4x4 matrices.
    Red = X, Green = Y, Blue = Z.
    """
    # 1. Adapt to input type: convert single matrix to a list
    if isinstance(matrices, (list, np.ndarray)) and np.array(matrices).ndim == 2:
        matrices = [matrices]
    
    geometries = []
    
    # 2. Add a World Frame at the origin for reference
    world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
        size=axis_size, origin=[0, 0, 0]
    )
    geometries.append(world_frame)
    geometries.append(mesh)
    
    # 3. Create and transform a coordinate frame for each matrix
    for i, mat in enumerate(matrices):
        # Create a frame
        pose_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
            size=axis_size, origin=[0, 0, 0]
        )
        # Apply the transformation
        pose_frame.transform(mat)
        geometries.append(pose_frame)
        
        print(f"Frame {i} Translation: {mat[:3, 3]}")

    # 4. Visualize everything together
    print(f"Visualizing {len(matrices)} transformation(s)...")
    o3d.visualization.draw_geometries(
        geometries,
        window_name="Adaptive Pose Visualization",
        width=640, height=480
    )

def visualize_from_pose(geometries, camera_pose):
    """
    Visualize scene from a given 4x4 camera pose matrix.
    Assumes pose = camera-to-world transform.
    """

    vis = o3d.visualization.Visualizer()
    vis.create_window()

    for g in geometries:
        vis.add_geometry(g)

    ctr = vis.get_view_control()

    # --- Extract camera position and rotation ---
    cam_pos = camera_pose[:3, 3]
    R_cam = camera_pose[:3, :3]   # camera-to-world rotation

    # --- Build Open3D extrinsic (world-to-camera) ---
    extrinsic = np.eye(4)
    extrinsic[:3, :3] = R_cam.T
    extrinsic[:3, 3] = -R_cam.T @ cam_pos

    # --- Apply ---
    param = ctr.convert_to_pinhole_camera_parameters()
    param.extrinsic = extrinsic
    ctr.convert_from_pinhole_camera_parameters(param)

    vis.run()
    vis.destroy_window()

SOURCE_PATH = "workpiece/first_object.stl" #  CAD model
mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals() # Makes the shading look better
R_mat = mesh.get_rotation_matrix_from_axis_angle([0, 0, np.radians(90)])
mesh.rotate(R_mat, center=[0, 0, 0])

# target_pose_0   = [200, 0, 300, 0, -135, 0]
# target_pose_1   = [200, 0, 300, 15, -135, 0]
# target_pose_2   = [200, 0, 300, -15, -135, 0]

target_pose_0   = [0, 200, 300, 135, 0, 0]
target_pose_1   = [0, 200, 300, 135, 15, 0]
target_pose_2   = [0, 200, 300, 135, -15, 0]

target_0 = pose_to_matrix(target_pose_0)
target_1 = pose_to_matrix(target_pose_1)
target_2 = pose_to_matrix(target_pose_2)

np.save(f"viewpoints_candidate/testing/viewpoint_pose_0.npy", target_0)
np.save(f"viewpoints_candidate/testing/viewpoint_pose_1.npy", target_1)
np.save(f"viewpoints_candidate/testing/viewpoint_pose_2.npy", target_2)

# target = np.array([[1, 0, 0, 0],
#                   [0, 1, 0, 300],
#                   [0, 0, 1, 300],
#                   [0, 0, 0, 1]])

target = target_2

visualize_transforms([target])
visualize_from_pose([mesh], target)

Frame 0 Translation: [  0. 200. 300.]
Visualizing 1 transformation(s)...


In [13]:
import open3d as o3d

# Load the point cloud
# pcd = o3d.io.read_point_cloud("viewpoints_candidate\testing_data\multiview_new_hemisphere_400_mix\viewpoint_simulated_0.pcd")
pcd = o3d.io.read_point_cloud(f"./viewpoints_candidate/testing_data/multiview_new_hemisphere_600_mix/viewpoint_simulated_1.pcd")
o3d.visualization.draw_geometries([pcd])